In [ ]:
# ============================================================================
# CACHE REDIRECTION -- must run BEFORE importing transformers/datasets/vllm
# ============================================================================
import os
import sys

os.environ["VLLM_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_MOE_USE_DEEP_GEMM"] = "0"
os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"

HF_CACHE_ROOT = "/work/hdd/bfrc"
os.environ["HF_HOME"]            = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_HUB_CACHE"]       = f"{HF_CACHE_ROOT}/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = f"{HF_CACHE_ROOT}/hf"
os.environ["HF_DATASETS_CACHE"]  = f"{HF_CACHE_ROOT}/hf/datasets"
os.environ["VLLM_CACHE_ROOT"]    = f"{HF_CACHE_ROOT}/vllm"
os.environ["TRITON_CACHE_DIR"]   = f"{HF_CACHE_ROOT}/triton"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = f"{HF_CACHE_ROOT}/torch_inductor"
os.environ["TMPDIR"]             = f"{HF_CACHE_ROOT}/tmp"
for p in (os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"],
          os.environ["HF_DATASETS_CACHE"], os.environ["VLLM_CACHE_ROOT"],
          os.environ["TRITON_CACHE_DIR"], os.environ["TORCHINDUCTOR_CACHE_DIR"],
          os.environ["TMPDIR"]):
    os.makedirs(p, exist_ok=True)

import ast, csv, gc, io, json, random, re, time, urllib.request
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd
from datasets import load_dataset
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import torch

# ============================================================================
# CONFIG
# ============================================================================
OUTPUT_DIR = "./outputs_5234__"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SMALL_MODELS = [
    {"name": "qwen2.5-7b",      "hf_id": "Qwen/Qwen2.5-7B-Instruct"},
    {"name": "gemma3-12b",      "hf_id": "google/gemma-3-12b-it"},
    {"name": "llama3.2-3b",     "hf_id": "meta-llama/Llama-3.2-3B-Instruct"},
    {"name": "mistral-7b-v0.3", "hf_id": "mistralai/Mistral-7B-Instruct-v0.3"},
]

SAMPLES_PER_MODEL = 5
TEMPERATURE       = 0.6
TOP_P             = 0.95
MAX_NEW_TOKENS    = 1024
MAX_MODEL_LEN     = 8192
GPU_MEM_UTIL      = 0.90
TENSOR_PARALLEL   = 1

# All three MuSR domains are run separately so per-domain accuracy is reported.
DATASETS_TO_RUN = ["musr_mm", "musr_op", "musr_ta", "folio"]

#   SARA: 272 -> 222 eval after 50 holdout
#   GPQA Diamond: 198 -> 148 eval after 50 holdout
#   MuSR murder_mysteries: 250 -> 200 eval after 50 holdout
#   MuSR object_placements: 256 -> 206 eval after 50 holdout
#   MuSR team_allocation: 250 -> 200 eval after 50 holdout
#   FOLIO validation: 203 -> 153 eval after 50 holdout
HOLDOUT_SIZES = {
    "sara":    50,
    "argkp":   8,
    "gpqa":    50,
    "musr_mm": 50,
    "musr_op": 50,
    "musr_ta": 50,
    "folio":   50,
}
ARGKP_CAP    = 200
HOLDOUT_SEED = 42
SEED_BASE    = 5234

# Maps the short dataset name used in DATASETS_TO_RUN to the HF split name.
MUSR_DOMAIN_BY_KEY = {
    "musr_mm": "murder_mysteries",
    "musr_op": "object_placements",
    "musr_ta": "team_allocation",
}

# ============================================================================
# PROMPTS
# ============================================================================
SHARED_SYSTEM = (
    "You are a careful reasoner. For every question, you must:\n"
    "1. Think step by step inside a single <reasoning>...</reasoning> block.\n"
    "2. Put ONLY the final answer (no explanation) inside a <answer>...</answer> tag.\n"
    "Do not write anything outside these two tags. Do not abbreviate your reasoning."
)

def sara_prompt(ex):
    return (
        "You are reasoning about US federal tax law.\n\n"
        f"STATUTE:\n{ex['statute']}\n\n"
        f"CASE SCENARIO:\n{ex['case']}\n\n"
        f"HYPOTHESIS:\n{ex['question']}\n\n"
        "TASK: Decide whether the statute, applied to the case scenario, ENTAILS or "
        "CONTRADICTS the hypothesis. Walk through:\n"
        "  (a) which statutory provisions apply to the case,\n"
        "  (b) what facts in the scenario trigger or fail to trigger those provisions,\n"
        "  (c) what the provisions imply about the hypothesis.\n\n"
        "Output exactly:\n"
        "<reasoning>your step-by-step legal analysis</reasoning>\n"
        "<answer>ENTAILED</answer>   OR   <answer>CONTRADICTED</answer>"
    )

def argkp_prompt(ex):
    stance_word = "SUPPORTS" if ex["stance"] == 1 else "OPPOSES"
    return (
        "You will defend a position on a controversial topic.\n\n"
        f"TOPIC: {ex['topic']}\n"
        f"STANCE: You must argue a position that {stance_word} this topic.\n\n"
        "TASK: Build the strongest case you can for this stance. Walk through:\n"
        "  (a) the core claim,\n"
        "  (b) two to four supporting key points, each with brief justification,\n"
        "  (c) how these points jointly defend the stance.\n\n"
        "Do not hedge or argue the other side. Stay on the assigned stance.\n\n"
        "Output exactly:\n"
        "<reasoning>your argument, structured as above</reasoning>\n"
        f"<answer>{stance_word}</answer>"
    )

def gpqa_prompt(ex):
    opts = ex["options"]
    block = "\n".join(f"  {L}. {t}" for L, t in zip("ABCD", opts))
    return (
        "You are answering a graduate-level science question. Exactly one option is correct.\n\n"
        f"QUESTION:\n{ex['question']}\n\n"
        f"OPTIONS:\n{block}\n\n"
        "TASK: Reason carefully through the underlying science. Walk through:\n"
        "  (a) what principles or equations the question is testing,\n"
        "  (b) how to apply them to the specifics of this problem,\n"
        "  (c) why one option follows and the others do not.\n\n"
        "Output exactly:\n"
        "<reasoning>your step-by-step scientific reasoning</reasoning>\n"
        "<answer>A</answer>   (or B, C, or D -- a single capital letter only)"
    )

def musr_prompt(ex):
    """MuSR narrative reasoning prompt. Supports up to 7 options (A-G)."""
    block = "\n".join(f"  {L}. {t}" for L, t in zip("ABCDEFG", ex["choices"]))
    last_letter = chr(ord('A') + len(ex["choices"]) - 1)
    return (
        "You are reasoning about a narrative scenario. Read the narrative carefully, "
        "then choose the single correct answer.\n\n"
        f"NARRATIVE:\n{ex['narrative']}\n\n"
        f"QUESTION:\n{ex['question']}\n\n"
        f"OPTIONS:\n{block}\n\n"
        "TASK: Reason carefully through the evidence in the narrative. Walk through:\n"
        "  (a) what facts the narrative establishes,\n"
        "  (b) how those facts bear on each candidate option,\n"
        "  (c) why one option is best supported and the others are not.\n\n"
        "Output exactly:\n"
        "<reasoning>your step-by-step reasoning over the narrative</reasoning>\n"
        f"<answer>A</answer>   (or B, C, ... up to {last_letter} -- a single capital letter only)"
    )

def folio_prompt(ex):
    return (
        "You are reasoning about first-order logical entailment in natural language.\n\n"
        f"PREMISES:\n{ex['premises']}\n\n"
        f"CONCLUSION:\n{ex['conclusion']}\n\n"
        "TASK: Decide whether the conclusion is logically entailed by the premises. "
        "Use these labels:\n"
        "  TRUE      -- the conclusion follows necessarily from the premises\n"
        "  FALSE     -- the negation of the conclusion follows from the premises\n"
        "  UNCERTAIN -- neither the conclusion nor its negation follows\n\n"
        "Walk through:\n"
        "  (a) what each premise asserts,\n"
        "  (b) what chain of inferences (if any) the conclusion would require,\n"
        "  (c) whether the premises are sufficient to establish or refute it.\n\n"
        "Output exactly:\n"
        "<reasoning>your step-by-step logical analysis</reasoning>\n"
        "<answer>TRUE</answer>   OR   <answer>FALSE</answer>   OR   <answer>UNCERTAIN</answer>"
    )

# All three MuSR domains share the same prompt builder.
PROMPT_BUILDERS = {
    "sara":    sara_prompt,
    "argkp":   argkp_prompt,
    "gpqa":    gpqa_prompt,
    "musr_mm": musr_prompt,
    "musr_op": musr_prompt,
    "musr_ta": musr_prompt,
    "folio":   folio_prompt,
}

# ============================================================================
# DATASET LOADERS
# ============================================================================
def _split(examples, holdout_size, seed):
    rng = random.Random(seed)
    idx = list(range(len(examples)))
    rng.shuffle(idx)
    hold = set(idx[:holdout_size])
    eval_set = [ex for i, ex in enumerate(examples) if i not in hold]
    holdout  = [examples[i] for i in idx[:holdout_size]]
    return eval_set, holdout

def load_sara():
    ds = None
    last_err = None
    for hf_id, cfg, split in [
        ("nguha/legalbench", "sara_entailment", "test"),
        ("nguha/legalbench", "sara_entailment", "train"),
    ]:
        try:
            ds = load_dataset(hf_id, cfg, split=split)
            print(f"  loaded SARA from {hf_id}/{cfg}/{split}: {len(ds)}")
            break
        except Exception as e:
            last_err = e
    if ds is None:
        raise RuntimeError(f"Could not load SARA: {last_err}")
    examples = []
    for i, row in enumerate(ds):
        if "text" in row:
            statute, case = "", row["text"]
        else:
            statute, case = row.get("statute", ""), row.get("case", "")
        ans = row.get("answer", "").strip().lower()
        if ans in ("entailment", "entailed", "yes", "true"):
            gold = "ENTAILED"
        elif ans in ("contradiction", "contradicted", "no", "false"):
            gold = "CONTRADICTED"
        else:
            continue
        examples.append({
            "dataset": "sara", "id": f"sara_{i:04d}",
            "statute": statute, "case": case,
            "question": row.get("question", ""), "gold": gold,
        })
    return _split(examples, HOLDOUT_SIZES["sara"], HOLDOUT_SEED)

def load_argkp():
    base = "https://raw.githubusercontent.com/IBM/KPA_2021_shared_task/main/kpm_data"
    df = None
    last_err = None
    for split in ("train", "dev"):
        url = f"{base}/arguments_{split}.csv"
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                df = pd.read_csv(io.BytesIO(r.read()))
            print(f"  loaded ArgKP arguments_{split}.csv from IBM GitHub: {len(df)} rows")
            break
        except Exception as e:
            last_err = e
    if df is None:
        local = "./data/argkp_arguments.csv"
        if os.path.exists(local):
            df = pd.read_csv(local)
            print(f"  loaded ArgKP from local file: {len(df)} rows")
        else:
            raise RuntimeError(
                f"Could not load ArgKP: {last_err}\n"
                f"Manual fallback: download {base}/arguments_train.csv and save as {local}"
            )

    examples, seen = [], set()
    for _, row in df.iterrows():
        topic = str(row.get("topic", "")).strip()
        stance = row.get("stance")
        if not topic or stance is None: continue
        try:
            stance = int(stance)
        except (ValueError, TypeError):
            continue
        if stance not in (1, -1): continue
        key = (topic, stance)
        if key in seen: continue
        seen.add(key)
        examples.append({
            "dataset": "argkp", "id": f"argkp_{len(examples):04d}",
            "topic": topic, "stance": stance,
            "gold": "SUPPORTS" if stance == 1 else "OPPOSES",
        })
    print(f"  unique (topic, stance) pairs: {len(examples)}")

    if len(examples) > ARGKP_CAP:
        examples = random.Random(HOLDOUT_SEED).sample(examples, ARGKP_CAP)
        print(f"  capped ArgKP to {ARGKP_CAP} (topic, stance) pairs")
    return _split(examples, HOLDOUT_SIZES["argkp"], HOLDOUT_SEED)

def load_gpqa():
    ds = None
    last_err = None
    for hf_id, cfg, split in [
        ("Idavidrein/gpqa", "gpqa_diamond", "train"),
        ("Idavidrein/gpqa", "gpqa_diamond", "test"),
    ]:
        try:
            ds = load_dataset(hf_id, cfg, split=split)
            print(f"  loaded GPQA from {hf_id}/{cfg}/{split}: {len(ds)}")
            break
        except Exception as e:
            last_err = e
    if ds is None:
        raise RuntimeError(
            f"Could not load GPQA Diamond: {last_err}\n"
            f"GPQA is gated -- accept license at https://huggingface.co/datasets/Idavidrein/gpqa "
            f"and run `huggingface-cli login`."
        )
    examples = []
    for i, row in enumerate(ds):
        q = row.get("Question") or row.get("question", "")
        correct = (row.get("Correct Answer") or row.get("correct_answer", "")).strip()
        wrongs = [(row.get(f"Incorrect Answer {k}") or row.get(f"incorrect_answer_{k}", "")).strip()
                  for k in (1, 2, 3)]
        wrongs = [w for w in wrongs if w]
        if not correct or len(wrongs) != 3: continue
        options = [correct] + wrongs
        rng = random.Random(hash(("gpqa", i, q[:50])) & 0xFFFFFFFF)
        rng.shuffle(options)
        correct_letter = "ABCD"[options.index(correct)]
        examples.append({
            "dataset": "gpqa", "id": f"gpqa_{i:04d}",
            "question": q, "options": options,
            "correct_letter": correct_letter, "gold": correct_letter,
            "subdomain": row.get("Subdomain") or row.get("subdomain", ""),
        })
    return _split(examples, HOLDOUT_SIZES["gpqa"], HOLDOUT_SEED)

def make_musr_loader(domain_key):
    """Factory that returns a loader for a specific MuSR domain.

    domain_key is one of 'musr_mm', 'musr_op', 'musr_ta'. The corresponding
    HF split is looked up in MUSR_DOMAIN_BY_KEY.
    """
    hf_split = MUSR_DOMAIN_BY_KEY[domain_key]

    def _loader():
        ds = None
        last_err = None
        for hf_id, split in [("TAUR-Lab/MuSR", hf_split)]:
            try:
                ds = load_dataset(hf_id, split=split)
                print(f"  loaded MuSR from {hf_id}/{split}: {len(ds)}")
                break
            except Exception as e:
                last_err = e
        if ds is None:
            raise RuntimeError(f"Could not load MuSR {hf_split}: {last_err}")

        examples = []
        for i, row in enumerate(ds):
            narrative = row.get("narrative", "")
            question = row.get("question", "")
            raw_choices = row.get("choices", "[]")
            if isinstance(raw_choices, str):
                try:
                    choices = json.loads(raw_choices)
                except json.JSONDecodeError:
                    try:
                        choices = ast.literal_eval(raw_choices)
                    except (ValueError, SyntaxError):
                        continue
            else:
                choices = list(raw_choices)
            if not choices or len(choices) > 7:
                continue
            idx = row.get("answer_index", None)
            if idx is None:
                ans_text = (row.get("answer_choice") or row.get("answer", "")).strip()
                if ans_text in choices:
                    idx = choices.index(ans_text)
                else:
                    continue
            try:
                idx = int(idx)
            except (TypeError, ValueError):
                continue
            if not (0 <= idx < len(choices)):
                continue
            gold_letter = "ABCDEFG"[idx]
            examples.append({
                "dataset": domain_key,
                "id": f"{domain_key}_{i:04d}",
                "narrative": narrative,
                "question": question,
                "choices": choices,
                "gold": gold_letter,
                "domain": hf_split,
            })
        return _split(examples, HOLDOUT_SIZES[domain_key], HOLDOUT_SEED)

    return _loader

def load_folio():
    ds = None
    last_err = None
    for hf_id, split in [
        ("tasksource/folio", "validation"),
        ("tasksource/folio", "train"),
        ("yale-nlp/FOLIO",   "validation"),
    ]:
        try:
            ds = load_dataset(hf_id, split=split)
            print(f"  loaded FOLIO from {hf_id}/{split}: {len(ds)}")
            break
        except Exception as e:
            last_err = e
    if ds is None:
        raise RuntimeError(f"Could not load FOLIO: {last_err}")

    examples = []
    for i, row in enumerate(ds):
        premises = (row.get("premises") or "").strip()
        conclusion = (row.get("conclusion") or "").strip()
        label = (row.get("label") or "").strip().lower()
        if not premises or not conclusion:
            continue
        if label in ("true", "entailment", "yes"):
            gold = "TRUE"
        elif label in ("false", "contradiction", "no"):
            gold = "FALSE"
        elif label in ("uncertain", "unknown", "neutral"):
            gold = "UNCERTAIN"
        else:
            continue
        examples.append({
            "dataset": "folio",
            "id": f"folio_{i:04d}",
            "premises": premises,
            "conclusion": conclusion,
            "gold": gold,
        })
    return _split(examples, HOLDOUT_SIZES["folio"], HOLDOUT_SEED)

LOADERS = {
    "sara":    load_sara,
    "argkp":   load_argkp,
    "gpqa":    load_gpqa,
    "musr_mm": make_musr_loader("musr_mm"),
    "musr_op": make_musr_loader("musr_op"),
    "musr_ta": make_musr_loader("musr_ta"),
    "folio":   load_folio,
}

# ============================================================================
# OUTPUT PARSING
# ============================================================================
_REASONING_RE = re.compile(r"<reasoning>(.*?)</reasoning>", re.DOTALL | re.IGNORECASE)
_ANSWER_RE    = re.compile(r"<answer>(.*?)</answer>",       re.DOTALL | re.IGNORECASE)

# All three MuSR domains share the same parsing logic (single capital letter A-G).
MUSR_KEYS = {"musr_mm", "musr_op", "musr_ta"}

def parse_output(raw, dataset):
    rm = _REASONING_RE.search(raw)
    am = _ANSWER_RE.search(raw)
    reasoning = rm.group(1).strip() if rm else None
    answer    = am.group(1).strip() if am else None

    if answer is None:
        tail = raw[-200:].upper()
        if dataset == "sara":
            if "ENTAILED" in tail or "ENTAILMENT" in tail: answer = "ENTAILED"
            elif "CONTRADICTED" in tail or "CONTRADICTION" in tail: answer = "CONTRADICTED"
        elif dataset == "argkp":
            if "SUPPORT" in tail: answer = "SUPPORTS"
            elif "OPPOSE" in tail or "AGAINST" in tail: answer = "OPPOSES"
        elif dataset == "gpqa":
            m = re.search(r"\b([ABCD])\b", tail)
            if m: answer = m.group(1)
        elif dataset in MUSR_KEYS:
            m = re.search(r"\b([ABCDEFG])\b", tail)
            if m: answer = m.group(1)
        elif dataset == "folio":
            if "UNCERTAIN" in tail or "UNKNOWN" in tail: answer = "UNCERTAIN"
            elif "TRUE" in tail or "ENTAIL" in tail: answer = "TRUE"
            elif "FALSE" in tail or "CONTRADICT" in tail: answer = "FALSE"

    if answer:
        answer = answer.strip().upper()
        if dataset == "sara":
            if answer.startswith("ENTAIL"):     answer = "ENTAILED"
            elif answer.startswith("CONTRADICT"): answer = "CONTRADICTED"
        elif dataset == "argkp":
            if answer.startswith("SUPP") or answer.startswith("PRO"):   answer = "SUPPORTS"
            elif answer.startswith("OPP") or answer.startswith("CON") or answer.startswith("AGAINST"):
                answer = "OPPOSES"
        elif dataset == "gpqa":
            if answer and answer[0] in "ABCD": answer = answer[0]
        elif dataset in MUSR_KEYS:
            if answer and answer[0] in "ABCDEFG": answer = answer[0]
        elif dataset == "folio":
            if answer.startswith("UNCERT") or answer.startswith("UNKNOWN"):
                answer = "UNCERTAIN"
            elif answer.startswith("TRUE") or answer.startswith("ENTAIL"):
                answer = "TRUE"
            elif answer.startswith("FALSE") or answer.startswith("CONTRADICT"):
                answer = "FALSE"

    if reasoning is None and raw.strip():
        reasoning = raw.strip()
    return {"reasoning": reasoning, "answer": answer,
            "parse_ok": (rm is not None) and (am is not None)}

# ============================================================================
# CSV SCHEMA + ROW BUILDER
# ============================================================================
CSV_FIELDS = [
    "question_id", "dataset", "split", "model", "sample_idx",
    "topic", "stance", "statute", "case", "hypothesis",
    "question_text", "option_a", "option_b", "option_c", "option_d", "subdomain",
    "narrative", "musr_domain", "choices_json",
    "premises", "conclusion",
    "gold_label", "raw_output", "reasoning", "predicted_label",
    "parse_ok", "seed", "wall_seconds",
]

def row_from_example(ex, split):
    base = {f: "" for f in CSV_FIELDS}
    base.update({"question_id": ex["id"], "dataset": ex["dataset"],
                 "split": split, "gold_label": ex["gold"]})
    if ex["dataset"] == "sara":
        base["statute"]    = ex.get("statute", "")
        base["case"]       = ex.get("case", "")
        base["hypothesis"] = ex.get("question", "")
    elif ex["dataset"] == "argkp":
        base["topic"]  = ex.get("topic", "")
        base["stance"] = ex.get("stance", "")
    elif ex["dataset"] == "gpqa":
        base["question_text"] = ex.get("question", "")
        opts = ex.get("options", [""]*4)
        base["option_a"], base["option_b"], base["option_c"], base["option_d"] = opts
        base["subdomain"] = ex.get("subdomain", "")
    elif ex["dataset"] in MUSR_KEYS:
        base["narrative"]    = ex.get("narrative", "")
        base["question_text"] = ex.get("question", "")
        base["choices_json"] = json.dumps(ex.get("choices", []))
        base["musr_domain"]  = ex.get("domain", "")
    elif ex["dataset"] == "folio":
        base["premises"]   = ex.get("premises", "")
        base["conclusion"] = ex.get("conclusion", "")
    return base

def already_done_keys(csv_path):
    done = set()
    if not os.path.exists(csv_path): return done
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                done.add((row["question_id"], row["model"], int(row["sample_idx"])))
            except (KeyError, ValueError): pass
    return done

# ============================================================================
# LOAD DATASETS FIRST so we fail fast before spinning up vLLM
# ============================================================================
print("[load] datasets")
ds_examples = {}
for ds_name in DATASETS_TO_RUN:
    print(f"  {ds_name}")
    eval_set, holdout = LOADERS[ds_name]()
    print(f"    eval: {len(eval_set)}   holdout: {len(holdout)}")
    ds_examples[ds_name] = (eval_set, holdout)

# ============================================================================
# MAIN LOOP
# ============================================================================
for model_spec in SMALL_MODELS:
    model_name = model_spec["name"]
    print(f"\n{'='*60}\n[model] {model_name}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_spec["hf_id"], trust_remote_code=True)
    llm = LLM(
        model=model_spec["hf_id"],
        dtype="bfloat16",
        trust_remote_code=True,
        gpu_memory_utilization=GPU_MEM_UTIL,
        max_model_len=MAX_MODEL_LEN,
        tensor_parallel_size=TENSOR_PARALLEL,
    )

    tasks = []
    for ds_name, (eval_set, holdout) in ds_examples.items():
        for split_name, examples in [("eval", eval_set), ("holdout", holdout)]:
            csv_path = os.path.join(OUTPUT_DIR, f"{ds_name}_{split_name}_cots.csv")
            done = already_done_keys(csv_path)
            for ex in examples:
                missing_samples = [s for s in range(SAMPLES_PER_MODEL)
                                   if (ex["id"], model_name, s) not in done]
                if not missing_samples: continue
                user_msg = PROMPT_BUILDERS[ds_name](ex)
                messages = [{"role": "system", "content": SHARED_SYSTEM},
                            {"role": "user",   "content": user_msg}]
                try:
                    prompt_str = tokenizer.apply_chat_template(
                        messages, add_generation_prompt=True, tokenize=False)
                except Exception:
                    messages = [{"role": "user", "content": f"{SHARED_SYSTEM}\n\n{user_msg}"}]
                    prompt_str = tokenizer.apply_chat_template(
                        messages, add_generation_prompt=True, tokenize=False)
                tasks.append({
                    "ds_name": ds_name, "split": split_name, "ex": ex,
                    "prompt": prompt_str, "csv_path": csv_path,
                    "missing_samples": missing_samples,
                })

    print(f"  [{model_name}] {len(tasks)} prompts to run")
    if not tasks:
        del llm, tokenizer; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    flat_prompts, flat_meta, flat_params = [], [], []
    for t in tasks:
        for sample_idx in t["missing_samples"]:
            seed = SEED_BASE + sample_idx
            flat_prompts.append(t["prompt"])
            flat_meta.append({**t, "sample_idx": sample_idx, "seed": seed})
            flat_params.append(SamplingParams(
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_NEW_TOKENS,
                seed=seed,
                n=1,
            ))

    print(f"  [{model_name}] dispatching {len(flat_prompts)} generations to vLLM")
    t0 = time.time()
    outputs = llm.generate(flat_prompts, flat_params)
    wall = time.time() - t0
    per_prompt = wall / max(1, len(flat_prompts))
    print(f"  [{model_name}] done in {wall:.1f}s ({per_prompt:.2f}s/prompt avg)")

    writers, files = {}, {}
    try:
        for meta, out in zip(flat_meta, outputs):
            csv_path = meta["csv_path"]
            if csv_path not in writers:
                is_new = not os.path.exists(csv_path)
                f = open(csv_path, "a", newline="", encoding="utf-8")
                w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
                if is_new: w.writeheader()
                writers[csv_path] = w; files[csv_path] = f
            raw = out.outputs[0].text
            parsed = parse_output(raw, meta["ds_name"])
            row = row_from_example(meta["ex"], meta["split"])
            row.update({
                "model": model_name,
                "sample_idx": meta["sample_idx"],
                "raw_output": raw,
                "reasoning": parsed["reasoning"] or "",
                "predicted_label": parsed["answer"] or "",
                "parse_ok": parsed["parse_ok"],
                "seed": meta["seed"],
                "wall_seconds": f"{per_prompt:.2f}",
            })
            writers[csv_path].writerow(row)
            files[csv_path].flush()
    finally:
        for f in files.values(): f.close()

    del llm, tokenizer, outputs
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\n[done] all CSVs written to", OUTPUT_DIR)
for p in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, p)
    n = sum(1 for _ in open(fp)) - 1
    print(f"  {p}: {n} rows")